# Transformer Baseline for ECG Arrhythmia Classification

A small Transformer encoder applied to the same inputs as the CNN baseline
(raw 1D beat waveforms, z-scored, SMOTE-balanced training set).

**Design goal:** parameter count comparable to the CNN baseline (~44k) so the
CNN vs. Transformer comparison is apples-to-apples. A larger variant can be
added afterwards if the matched-size version is promising.

**Pipeline:**
1. Load `X_train_smote`, `X_val`, `X_test` (same `.npy` files the CNN uses).
2. Per-beat z-score.
3. Patch-embed the 250-sample waveform into a short token sequence.
4. Prepend a learnable `[CLS]` token, add learnable positional embeddings.
5. Run a small Transformer encoder; classify from the `[CLS]` token.
6. Random hyperparameter search (15 configs, same budget as CNN).
7. Final training on best config, evaluation on test, attention visualization.

In [1]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import random
import time
from collections import Counter

ModuleNotFoundError: No module named 'torch'

## Load the dataset

In [ ]:
# Output directory for model artifacts
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

In [ ]:
# Training data — SMOTE-balanced
X_train = np.load('../Data/processed/X_train_smote.npy')
y_train = np.load('../Data/processed/y_train_smote.npy')

# Validation and test — original splits, NO SMOTE
X_val = np.load('../Data/processed/X_val.npy')
y_val = np.load('../Data/processed/y_val.npy')
X_test = np.load('../Data/processed/X_test.npy')
y_test = np.load('../Data/processed/y_test.npy')

print(f'Train: {X_train.shape}, {y_train.shape}')
print(f'Val:   {X_val.shape}, {y_val.shape}')
print(f'Test:  {X_test.shape}, {y_test.shape}')
print(f'Train class distribution: {dict(Counter(y_train.tolist()))}')

In [ ]:
# --- Global configuration ---
USE_SMOTE   = True
EPOCHS      = 50
PATIENCE    = 5
NUM_CLASSES = len(np.unique(y_train))
SEED        = 42
LABEL_NAMES = ['N (Normal)', 'S (Supravent.)', 'V (Ventric.)', 'F (Fusion)']

## Z-score normalization

Same as the CNN: normalize each beat independently so the model sees shape,
not patient-specific amplitude.

In [ ]:
def zscore(x):
    mean = x.mean(axis=1, keepdims=True)
    std  = x.std(axis=1, keepdims=True) + 1e-8
    return (x - mean) / std

X_train = zscore(X_train).astype(np.float32)
X_val   = zscore(X_val).astype(np.float32)
X_test  = zscore(X_test).astype(np.float32)

y_train = y_train.astype(np.int64)
y_val   = y_val.astype(np.int64)
y_test  = y_test.astype(np.int64)

# Transformer expects (batch, length) — no channel dim; the patch embed adds it
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

## Device and DataLoader helper

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Device: {device}')

def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

## Transformer model

The architecture is intentionally small so the parameter count is in the same
ballpark as the CNN baseline (~44k trainable params).

**Components:**
- `PatchEmbed1D` — splits the 250-sample waveform into non-overlapping patches
  of length `patch_size` and projects each to `d_model`. With `patch_size=10`
  we get 25 tokens per beat.
- Learnable `[CLS]` token prepended to the sequence.
- Learnable positional embeddings (no sinusoidal variant — sequences are short).
- `TransformerEncoderLayerWithAttn` — a standard pre-norm encoder layer that
  also **returns the attention weights** so we can visualize them later.
- Classifier head on the final `[CLS]` representation.

In [ ]:
class PatchEmbed1D(nn.Module):
    """Split a 1D waveform into non-overlapping patches and project each to d_model."""

    def __init__(self, seq_len: int, patch_size: int, d_model: int):
        super().__init__()
        assert seq_len % patch_size == 0, \
            f'seq_len ({seq_len}) must be divisible by patch_size ({patch_size})'
        self.patch_size = patch_size
        self.n_patches  = seq_len // patch_size
        # Conv1d with stride=kernel_size is a clean way to do non-overlapping patches
        self.proj = nn.Conv1d(
            in_channels=1, out_channels=d_model,
            kernel_size=patch_size, stride=patch_size,
        )

    def forward(self, x):
        # x: (B, L) — add channel dim for Conv1d
        x = x.unsqueeze(1)            # (B, 1, L)
        x = self.proj(x)              # (B, d_model, n_patches)
        x = x.transpose(1, 2)         # (B, n_patches, d_model)
        return x


class TransformerEncoderLayerWithAttn(nn.Module):
    """Pre-norm Transformer encoder layer that exposes attention weights."""

    def __init__(self, d_model: int, n_heads: int, ff_dim: int, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, return_attn: bool = False):
        # Multi-head self-attention (pre-norm)
        h = self.norm1(x)
        attn_out, attn_w = self.attn(
            h, h, h, need_weights=return_attn, average_attn_weights=True,
        )
        x = x + self.drop(attn_out)
        # Feed-forward (pre-norm)
        x = x + self.drop(self.ff(self.norm2(x)))
        return x, attn_w


class ECGTransformer(nn.Module):
    """Small Transformer encoder for 1D ECG beat classification."""

    def __init__(
        self,
        seq_len: int = 250,
        patch_size: int = 10,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        ff_dim: int = 128,
        num_classes: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed1D(seq_len, patch_size, d_model)
        n_patches = self.patch_embed.n_patches

        # Learnable CLS token and positional embeddings (+1 for CLS)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.drop = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerEncoderLayerWithAttn(d_model, n_heads, ff_dim, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x, return_attn: bool = False):
        # x: (B, L)
        B = x.size(0)
        tokens = self.patch_embed(x)                           # (B, N, D)
        cls = self.cls_token.expand(B, -1, -1)                  # (B, 1, D)
        x = torch.cat([cls, tokens], dim=1)                     # (B, N+1, D)
        x = x + self.pos_embed
        x = self.drop(x)

        attn_maps = []
        for layer in self.layers:
            x, attn_w = layer(x, return_attn=return_attn)
            if return_attn:
                attn_maps.append(attn_w)

        x = self.norm(x)
        logits = self.head(x[:, 0])                             # CLS token
        if return_attn:
            return logits, attn_maps
        return logits


# Sanity check — build once and report parameter count
_tmp = ECGTransformer().to(device)
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}  (CNN baseline was 44,228)\n')
print(_tmp)
del _tmp

## Training / evaluation helper

One-epoch function identical in shape to the CNN notebook — returns loss,
accuracy, macro-F1, and the true/predicted label arrays for downstream
reporting.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_samples, total_correct = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            outputs = model(xb)
            loss    = criterion(outputs, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = outputs.argmax(dim=1)
            total_loss    += loss.item() * xb.size(0)
            total_samples += yb.size(0)
            total_correct += (preds == yb).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    f1m      = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, accuracy, f1m, np.array(all_labels), np.array(all_preds)

## Hyperparameter tuning

Random search over 15 configurations, same budget as the CNN baseline for a
fair comparison. Model selection uses **macro F1** on the validation set
(accuracy is misleading when the majority class dominates).

Search space:
- Learning rate  
- Batch size  
- `d_model` / `n_layers` — capacity knobs  
- `patch_size` — how finely the waveform is tokenized  
- Dropout, weight decay

In [ ]:
LR_GRID         = [1e-4, 5e-4, 1e-3, 3e-3]
BATCH_SIZE_GRID = [64, 128]
D_MODEL_GRID    = [48, 64]
N_LAYERS_GRID   = [2, 3]
PATCH_SIZE_GRID = [5, 10, 25]   # 50, 25, 10 tokens respectively
DROPOUT_GRID    = [0.1, 0.2]
WD_GRID         = [0, 1e-4, 1e-3]

N_RANDOM_SAMPLES = 15
TUNING_EPOCHS    = 15     # shorter than CNN's 25 — transformers converge faster here
TUNING_PATIENCE  = 4

full_space = list(itertools.product(
    LR_GRID, BATCH_SIZE_GRID, D_MODEL_GRID,
    N_LAYERS_GRID, PATCH_SIZE_GRID, DROPOUT_GRID, WD_GRID,
))
random.seed(42)
combos = random.sample(full_space, k=min(N_RANDOM_SAMPLES, len(full_space)))

print(f'Full search space size     : {len(full_space)}')
print(f'Sampled configurations     : {len(combos)}\n')
for lr, bs, dm, nl, ps, dp, wd in combos:
    print(f'  lr={lr:.0e}  bs={bs:>3}  d_model={dm}  layers={nl}  '
          f'patch={ps}  drop={dp}  wd={wd}')

In [ ]:
def train_one_config(lr, batch_size, d_model, n_layers, patch_size,
                     dropout, weight_decay, max_epochs, patience):
    torch.manual_seed(SEED)

    train_loader_cfg = make_loader(X_train, y_train, batch_size, shuffle=True)
    val_loader_cfg   = make_loader(X_val,   y_val,   batch_size, shuffle=False)

    # n_heads must divide d_model; 4 works for both 48 and 64
    model = ECGTransformer(
        seq_len=250, patch_size=patch_size,
        d_model=d_model, n_heads=4, n_layers=n_layers,
        ff_dim=d_model * 2, num_classes=NUM_CLASSES, dropout=dropout,
    ).to(device)

    if USE_SMOTE:
        criterion = nn.CrossEntropyLoss()
    else:
        classes = np.unique(y_train)
        weights = compute_class_weight('balanced', classes=classes, y=y_train)
        class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6,
    )

    best_val_loss = float('inf')
    best_val_f1   = 0.0
    best_val_acc  = 0.0
    no_improve    = 0
    used_epochs   = 0

    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        run_epoch(model, train_loader_cfg, criterion, optimizer)
        va_loss, va_acc, va_f1, _, _ = run_epoch(model, val_loader_cfg, criterion)
        scheduler.step(va_loss)
        used_epochs = epoch

        if va_loss < best_val_loss:
            best_val_loss = va_loss
            best_val_f1   = va_f1
            best_val_acc  = va_acc
            no_improve    = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        'val_loss': best_val_loss,
        'val_acc':  best_val_acc,
        'val_f1':   best_val_f1,
        'epochs':   used_epochs,
        'seconds':  time.time() - t0,
        'n_params': n_params,
    }

In [ ]:
results = []

print(f'Running {len(combos)} configurations...\n')
header = (
    f'{"#":>3} | {"lr":>7} | {"bs":>4} | {"dm":>3} | {"nl":>2} | '
    f'{"ps":>3} | {"drop":>5} | {"wd":>7} | {"params":>7} | '
    f'{"val loss":>9} | {"val acc":>8} | {"val F1":>7} | {"ep":>3} | {"time":>6}'
)
print(header)
print('-' * len(header))

for i, (lr, bs, dm, nl, ps, dp, wd) in enumerate(combos, 1):
    r = train_one_config(lr, bs, dm, nl, ps, dp, wd, TUNING_EPOCHS, TUNING_PATIENCE)
    r.update({
        'lr': lr, 'batch_size': bs, 'd_model': dm, 'n_layers': nl,
        'patch_size': ps, 'dropout': dp, 'weight_decay': wd,
    })
    results.append(r)
    print(f'{i:>3} | {lr:>7.0e} | {bs:>4} | {dm:>3} | {nl:>2} | '
          f'{ps:>3} | {dp:>5} | {wd:>7} | {r["n_params"]:>7,} | '
          f'{r["val_loss"]:>9.4f} | {r["val_acc"]:>8.4f} | '
          f'{r["val_f1"]:>7.4f} | {r["epochs"]:>3d} | {r["seconds"]:>5.0f}s')

tuning_df = pd.DataFrame(results)[[
    'lr', 'batch_size', 'd_model', 'n_layers', 'patch_size',
    'dropout', 'weight_decay', 'n_params',
    'val_loss', 'val_acc', 'val_f1', 'epochs', 'seconds',
]]
tuning_df = tuning_df.sort_values('val_f1', ascending=False).reset_index(drop=True)

csv_path = f'{MODELS_DIR}/transformer_tuning_results.csv'
tuning_df.to_csv(csv_path, index=False)
print(f'\nResults saved to: {csv_path}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pv1 = tuning_df.pivot_table(
    index='lr', columns='batch_size', values='val_f1', aggfunc='mean',
).sort_index(ascending=False)
sns.heatmap(pv1, annot=True, fmt='.4f', cmap='Blues', ax=axes[0],
            cbar_kws={'label': 'Mean val macro F1'})
axes[0].set_title('Learning rate × Batch size', fontweight='bold')
axes[0].set_xlabel('Batch size'); axes[0].set_ylabel('Learning rate')

pv2 = tuning_df.pivot_table(
    index='d_model', columns='patch_size', values='val_f1', aggfunc='mean',
)
sns.heatmap(pv2, annot=True, fmt='.4f', cmap='Blues', ax=axes[1],
            cbar_kws={'label': 'Mean val macro F1'})
axes[1].set_title('d_model × patch_size', fontweight='bold')
axes[1].set_xlabel('Patch size'); axes[1].set_ylabel('d_model')

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/transformer_tuning_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Top 5 configurations by validation macro F1:\n')
print(tuning_df.head(5).to_string(index=False))

In [ ]:
best = tuning_df.iloc[0]
BEST_LR         = float(best['lr'])
BEST_BS         = int(best['batch_size'])
BEST_D_MODEL    = int(best['d_model'])
BEST_N_LAYERS   = int(best['n_layers'])
BEST_PATCH_SIZE = int(best['patch_size'])
BEST_DROPOUT    = float(best['dropout'])
BEST_WD         = float(best['weight_decay'])

print('Best configuration:')
print(f'  Learning rate : {BEST_LR:.0e}')
print(f'  Batch size    : {BEST_BS}')
print(f'  d_model       : {BEST_D_MODEL}')
print(f'  n_layers      : {BEST_N_LAYERS}')
print(f'  patch_size    : {BEST_PATCH_SIZE}')
print(f'  Dropout       : {BEST_DROPOUT}')
print(f'  Weight decay  : {BEST_WD}')
print(f'  Params        : {int(best["n_params"]):,}')
print(f'  Val F1 macro  : {best["val_f1"]:.4f}')
print(f'  Val accuracy  : {best["val_acc"]:.4f}')

## Final training

Retrain the best configuration for the full epoch budget with early stopping,
save the best checkpoint, and evaluate on the held-out test set.

In [ ]:
torch.manual_seed(SEED)

train_loader = make_loader(X_train, y_train, BEST_BS, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   BEST_BS, shuffle=False)
test_loader  = make_loader(X_test,  y_test,  BEST_BS, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')
print(f'Test batches : {len(test_loader)}')

model = ECGTransformer(
    seq_len=250, patch_size=BEST_PATCH_SIZE,
    d_model=BEST_D_MODEL, n_heads=4, n_layers=BEST_N_LAYERS,
    ff_dim=BEST_D_MODEL * 2, num_classes=NUM_CLASSES, dropout=BEST_DROPOUT,
).to(device)

if USE_SMOTE:
    criterion = nn.CrossEntropyLoss()
else:
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(model.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6,
)

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
epochs_without_improvement = 0
ckpt_path = f'{MODELS_DIR}/transformer_baseline.pt'

print('Training...\n')
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, _, _, _ = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc, _, _, _ = run_epoch(model, val_loader,   criterion)
    scheduler.step(va_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        epochs_without_improvement += 1

    print(f'Epoch {epoch:3d}/{EPOCHS} | '
          f'train loss={tr_loss:.4f} acc={tr_acc:.4f} | '
          f'val loss={va_loss:.4f} acc={va_acc:.4f}')

    if epochs_without_improvement >= PATIENCE:
        print(f'\nEarly stopping: no improvement for {PATIENCE} epochs')
        break

print(f'\nBest val loss: {best_val_loss:.4f}')

In [ ]:
fig = plt.figure(figsize=(12, 5))
plt.plot(history['train_loss'], label='Train Loss', color='#4C72B0')
plt.plot(history['val_loss'],   label='Val Loss',   color='#DD8452')
plt.title('Transformer Baseline Training Curves')
plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy Loss')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 5))
plt.plot(history['train_acc'], label='Train Acc', color='#4C72B0')
plt.plot(history['val_acc'],   label='Val Acc',   color='#DD8452')
plt.title('Transformer Baseline Training Curves')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Test evaluation

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))

test_loss, test_acc, test_f1, y_true, y_pred = run_epoch(model, test_loader, criterion)

print(f'Test loss         : {test_loss:.4f}')
print(f'Test accuracy     : {test_acc:.4f}')
print(f'Test F1 (macro)   : {test_f1:.4f}\n')

report = classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4)
print(report)

with open(f'{MODELS_DIR}/transformer_baseline_report.txt', 'w') as f:
    f.write(f'Transformer Baseline (PyTorch) — use_smote={USE_SMOTE}\n')
    f.write(f'Best config: lr={BEST_LR:.0e}, batch_size={BEST_BS}, '
            f'd_model={BEST_D_MODEL}, n_layers={BEST_N_LAYERS}, '
            f'patch_size={BEST_PATCH_SIZE}, dropout={BEST_DROPOUT}, '
            f'weight_decay={BEST_WD}\n')
    f.write('=' * 60 + '\n\n')
    f.write(f'Test loss        : {test_loss:.4f}\n')
    f.write(f'Test accuracy    : {test_acc:.4f}\n')
    f.write(f'Test F1 (macro)  : {test_f1:.4f}\n\n')
    f.write(report)

np.save(f'{MODELS_DIR}/transformer_baseline_history.npy', history, allow_pickle=True)

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion matrix — counts', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            cbar_kws={'label': 'Row-normalized'})
axes[1].set_title('Confusion matrix — recall per class', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/transformer_baseline_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample predictions — green = correct, red = wrong
n_show = 8
idxs = random.sample(range(len(y_true)), n_show)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, i in zip(axes.flatten(), idxs):
    beat = X_test[i]
    true = LABEL_NAMES[y_true[i]]
    pred = LABEL_NAMES[y_pred[i]]
    color = '#55A868' if y_true[i] == y_pred[i] else '#C44E52'
    ax.plot(beat, color=color, linewidth=1.2)
    ax.set_title(f'True: {true}\nPred: {pred}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Sample test predictions — green = correct, red = wrong',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Attention visualization

For one beat from each class, plot the waveform and overlay the CLS-token's
attention weights onto the original sample positions. This shows **which
regions of the beat the Transformer looked at** to make its decision — the
kind of interpretability the project's research question asks about.

In [ ]:
model.eval()

# Pick the first correctly-classified sample from each class
sample_idxs = []
for c in range(NUM_CLASSES):
    correct_mask = (y_true == c) & (y_pred == c)
    candidates = np.where(correct_mask)[0]
    if len(candidates) > 0:
        sample_idxs.append((c, int(candidates[0])))

fig, axes = plt.subplots(len(sample_idxs), 1,
                         figsize=(12, 2.5 * len(sample_idxs)))
if len(sample_idxs) == 1:
    axes = [axes]

for ax, (cls, idx) in zip(axes, sample_idxs):
    beat = X_test[idx]
    x_in = torch.from_numpy(beat[np.newaxis, :]).to(device)
    with torch.no_grad():
        _, attn_maps = model(x_in, return_attn=True)

    # Use the LAST layer's attention from the CLS token to every patch
    last = attn_maps[-1][0]                # (tokens, tokens)
    cls_attn = last[0, 1:].cpu().numpy()    # drop self-attention on CLS

    # Expand per-patch weights onto the original 250 samples
    expanded = np.repeat(cls_attn, BEST_PATCH_SIZE)
    if len(expanded) < len(beat):
        expanded = np.pad(expanded, (0, len(beat) - len(expanded)))
    expanded = expanded[:len(beat)]

    ax.plot(beat, color='black', linewidth=1.0, label='beat')
    ax.fill_between(np.arange(len(beat)), beat.min(), beat.max(),
                    where=expanded > expanded.mean(),
                    alpha=0.2, color='#C44E52', label='high attention')
    ax2 = ax.twinx()
    ax2.plot(expanded, color='#C44E52', linewidth=0.8, alpha=0.7)
    ax2.set_ylabel('attention', color='#C44E52')
    ax.set_title(f'Class {LABEL_NAMES[cls]} — test sample #{idx}')
    ax.set_xlabel('sample index')
    ax.set_ylabel('amplitude (z-scored)')

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/transformer_attention.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
summary = tuning_df[[
    'lr', 'batch_size', 'd_model', 'n_layers', 'patch_size',
    'dropout', 'weight_decay', 'val_loss', 'val_acc', 'val_f1',
]].copy()
summary['lr'] = summary['lr'].apply(lambda x: f'{x:.0e}')
summary.columns = [
    'Learning rate', 'Batch size', 'd_model', 'n_layers', 'Patch size',
    'Dropout', 'Weight decay', 'Val loss', 'Val accuracy', 'Val macro F1',
]
print('Hyperparameter search results (sorted by val F1):')
print(summary.to_string(index=False))
print()
print('Best config retrained on full training budget:')
print(f'  Learning rate : {BEST_LR:.0e}')
print(f'  Batch size    : {BEST_BS}')
print(f'  d_model       : {BEST_D_MODEL}')
print(f'  n_layers      : {BEST_N_LAYERS}')
print(f'  Patch size    : {BEST_PATCH_SIZE}')
print(f'  Dropout       : {BEST_DROPOUT}')
print(f'  Weight decay  : {BEST_WD}')
print(f'  Test accuracy : {test_acc:.4f}')
print(f'  Test F1 macro : {test_f1:.4f}')